# 🌾 Bugesera Harvest Prediction System
## Machine Learning Model Training Notebook
**Author:** Cesalie UWIMPUHWE | Rwanda Polytechnic  
**Dataset:** Bugesera District Agricultural Data 2020–2024  
**Units:** ARE (1 ha = 100 are) | Target: kg/are

---
### Steps
1. Load & Explore Dataset  
2. Feature Engineering  
3. Encode Categorical Features  
4. Define Features & Target  
5. Train/Test Split & Scaling  
6. Train 3 Models (Linear Regression, Gradient Boosting, Random Forest)  
7. Compare & Select Best Model  
8. Save All Artifacts

In [ ]:
# STEP 1: IMPORT LIBRARIES
import pandas as pd
import numpy as np
import joblib, json, os, warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("✅ Libraries loaded")
print(f"scikit-learn version: {__import__('sklearn').__version__}")

In [ ]:
# STEP 1: LOAD DATASET
PATH = 'Bugesera_Agricultural_Dataset_2020_2024_Done.xlsx'

harvest = pd.read_excel(PATH, sheet_name='Harvest Records (2020-2024)')
climate = pd.read_excel(PATH, sheet_name='Climate Data (2020-2024)')
soil    = pd.read_excel(PATH, sheet_name='Soil Analysis')
pest    = pd.read_excel(PATH, sheet_name='Pest & Disease Records')
cost    = pd.read_excel(PATH, sheet_name='Farm Input Costs')

print(f"Harvest Records : {harvest.shape}")
print(f"Climate Data    : {climate.shape}")
print(f"Soil Analysis   : {soil.shape}")
print(f"Pest Records    : {pest.shape}")
print(f"Input Costs     : {cost.shape}")
print(f"\nCrops   : {sorted(harvest.Crop_Type.unique())}")
print(f"Seasons : {sorted(harvest.Season.unique())}")
print(f"Sectors : {sorted(harvest.Sector.unique())}")
harvest.head()

In [ ]:
# STEP 1: EXPLORE — Yield distribution by crop
print("=== Yield by Crop (kg/ha) ===")
print(harvest.groupby('Crop_Type')['Yield_Kg_per_Ha'].agg(['mean','std','min','max']).round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, crop in zip(axes, ['Maize','Beans','Rice']):
    data = harvest[harvest.Crop_Type == crop]['Yield_Kg_per_Ha']
    ax.hist(data, bins=20, color='#22c55e', edgecolor='white', alpha=0.8)
    ax.set_title(f'{crop} Yield Distribution', fontweight='bold')
    ax.set_xlabel('Yield (kg/ha)')
    ax.set_ylabel('Count')
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.0f}')
    ax.legend()
plt.tight_layout()
plt.savefig('yield_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Distribution plot saved")

In [ ]:
# STEP 2: FEATURE ENGINEERING

# 2a. Aggregate climate seasonally per sector
clim = climate.groupby(['Year','Season','Sector']).agg(
    Avg_Temperature_Celsius   = ('Avg_Temperature_Celsius','mean'),
    Total_Rainfall_mm         = ('Total_Rainfall_mm','sum'),
    Relative_Humidity_Percent = ('Relative_Humidity_Percent','mean'),
    Sunshine_Hours_per_Day    = ('Sunshine_Hours_per_Day','mean'),
    Wind_Speed_kmh            = ('Wind_Speed_kmh','mean'),
    Evapotranspiration_mm     = ('Evapotranspiration_mm','sum')
).reset_index()
print(f"Climate aggregated: {clim.shape}")

# 2b. Aggregate pest loss per sector/season
pest_agg = pest.groupby(['Year','Season','Sector']).agg(
    Avg_Pest_YieldLoss=('Yield_Loss_Percent','mean')
).reset_index()

# 2c. Add ARE unit columns (1 ha = 100 are)
harvest['Farm_Size_Are']                = (harvest['Farm_Size_Ha'] * 100).round(2)
harvest['Area_Planted_Are']             = (harvest['Area_Planted_Ha'] * 100).round(2)
harvest['Yield_Kg_per_Are']             = (harvest['Yield_Kg_per_Ha'] / 100).round(6)
harvest['Fertilizer_Amount_Kg_per_Are'] = (harvest['Fertilizer_Amount_Kg_per_Ha'] / 100).round(6)
cost['Total_Cost_RWF_per_Are']          = (cost['Total_Cost_RWF_per_Ha'] / 100).round(2)

# 2d. Merge all sheets
df = harvest.copy()
df = df.merge(clim, on=['Year','Season','Sector'], how='left')
df = df.merge(soil[['Sector','pH_Level','Organic_Matter_Percent',
                     'Nitrogen_ppm','Phosphorus_ppm','Potassium_ppm']], on='Sector', how='left')
df = df.merge(pest_agg, on=['Year','Season','Sector'], how='left')
df = df.merge(cost[['Year','Season','Sector','Crop_Type','Total_Cost_RWF_per_Are']],
              on=['Year','Season','Sector','Crop_Type'], how='left')

# 2e. Fill missing values
df['Avg_Pest_YieldLoss']     = df['Avg_Pest_YieldLoss'].fillna(df['Avg_Pest_YieldLoss'].median())
df['Pest_Disease_Pressure']  = df['Pest_Disease_Pressure'].fillna('Low')
df['Total_Cost_RWF_per_Are'] = df['Total_Cost_RWF_per_Are'].fillna(df['Total_Cost_RWF_per_Are'].median())

# 2f. Engineered features
fert_map = {'Yes': 1.0, 'Partial': 0.5, 'No': 0.0}
df['Fertilizer_Score']   = df['Fertilizer_Used'].map(fert_map).fillna(0.0)
df['Irrigation_Score']   = df['Irrigation_Used'].map({'Yes':1.0,'Partial':0.5,'No':0.0}).fillna(0.0)
df['Soil_pH_Optimality'] = 1 - abs(df['pH_Level'] - 6.5) / 2.0
crop_rain = {'Maize': 500, 'Beans': 400, 'Rice': 650}
df['Rain_Adequacy'] = df.apply(
    lambda r: min(r['Total_Rainfall_mm'] / crop_rain.get(r['Crop_Type'], 500), 1.5), axis=1)
df['Is_Season_A'] = (df['Season'] == 'Season A').astype(int)

print(f"\nFinal dataset shape: {df.shape}")
print(f"Missing values    : {df.isnull().sum().sum()}")
print(f"\nYield range (kg/are): {df.Yield_Kg_per_Are.min():.2f} – {df.Yield_Kg_per_Are.max():.2f}")
print(f"Yield mean  (kg/are): {df.Yield_Kg_per_Are.mean():.2f}")

In [ ]:
# STEP 3: ENCODE CATEGORICAL FEATURES
CAT_COLS = ['Season','Crop_Type','Sector','Farmer_Category',
            'Fertilizer_Used','Irrigation_Used','Soil_Health',
            'Previous_Crop','Weather_Impact','Pest_Disease_Pressure',
            'Quality_Grade','Labor_Availability','Extension_Service_Access','Credit_Access']

le_dict = {}
df_enc  = df.copy()
for col in CAT_COLS:
    if col in df_enc.columns:
        le = LabelEncoder()
        df_enc[col] = le.fit_transform(df_enc[col].astype(str))
        le_dict[col] = le
        print(f"  {col}: {le.classes_.tolist()}")

joblib.dump(le_dict, 'label_encoders.pkl')
print(f"\n✅ Encoded {len(le_dict)} categorical features")
print(f"   Saved → label_encoders.pkl")

In [ ]:
# STEP 4: DEFINE FEATURES AND TARGET
FEATURES = [
    'Year','Season','Crop_Type','Sector',
    'Farm_Size_Are','Area_Planted_Are',
    'Farmer_Category','Fertilizer_Used','Fertilizer_Amount_Kg_per_Are',
    'Irrigation_Used','Soil_Health','Previous_Crop',
    'Weather_Impact','Pest_Disease_Pressure',
    'Labor_Availability','Extension_Service_Access','Credit_Access',
    'Market_Distance_km',
    'Avg_Temperature_Celsius','Total_Rainfall_mm',
    'Relative_Humidity_Percent','Sunshine_Hours_per_Day',
    'Wind_Speed_kmh','Evapotranspiration_mm',
    'pH_Level','Organic_Matter_Percent',
    'Nitrogen_ppm','Phosphorus_ppm','Potassium_ppm',
    'Avg_Pest_YieldLoss','Total_Cost_RWF_per_Are',
    'Fertilizer_Score','Irrigation_Score',
    'Soil_pH_Optimality','Rain_Adequacy','Is_Season_A'
]

TARGET = 'Yield_Kg_per_Are'
FEATURES = [f for f in FEATURES if f in df_enc.columns]

X = df_enc[FEATURES].fillna(df_enc[FEATURES].median(numeric_only=True))
y = df_enc[TARGET]

print(f"Number of features : {len(FEATURES)}")
print(f"Target variable    : {TARGET}")
print(f"X shape            : {X.shape}")
print(f"y range            : {y.min():.2f} – {y.max():.2f} kg/are")
print(f"y mean             : {y.mean():.2f} kg/are")
print(f"\nFeature list:\n{FEATURES}")

In [ ]:
# STEP 5: TRAIN/TEST SPLIT AND SCALING
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")

# Standard scaler for Linear Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, 'scaler.pkl')
print(f"\n✅ Saved scaler.pkl")
print(f"   Scaler mean range: {scaler.mean_.min():.2f} – {scaler.mean_.max():.2f}")

In [ ]:
# STEP 6: TRAIN 3 MODELS

models = {
    'Linear Regression': LinearRegression(),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, random_state=42),
    'Random Forest'    : RandomForestRegressor(
        n_estimators=300, max_depth=12, min_samples_leaf=2,
        random_state=42, n_jobs=-1),
}

results = {}
trained = {}

for name, mdl in models.items():
    print(f"\nTraining: {name}...")
    if name == 'Linear Regression':
        mdl.fit(X_train_sc, y_train)
        pred_tr = mdl.predict(X_train_sc)
        pred_te = mdl.predict(X_test_sc)
    else:
        mdl.fit(X_train, y_train)
        pred_tr = mdl.predict(X_train)
        pred_te = mdl.predict(X_test)

    r2_tr = r2_score(y_train, pred_tr)
    r2_te = r2_score(y_test,  pred_te)
    mae   = mean_absolute_error(y_test, pred_te)
    rmse  = np.sqrt(mean_squared_error(y_test, pred_te))
    acc   = max(0, (1 - mae / y_test.mean())) * 100

    results[name] = {
        'r2_train': round(float(r2_tr),4), 'r2_test': round(float(r2_te),4),
        'mae': round(float(mae),4), 'rmse': round(float(rmse),4),
        'accuracy': round(float(acc),2)
    }
    trained[name] = mdl
    print(f"  R² Train : {r2_tr:.4f}")
    print(f"  R² Test  : {r2_te:.4f}")
    print(f"  MAE      : {mae:.4f} kg/are")
    print(f"  RMSE     : {rmse:.4f} kg/are")
    print(f"  Accuracy : {acc:.1f}%")

# Save individual models
joblib.dump(trained['Random Forest'],     'random_forest.pkl')
joblib.dump(trained['Gradient Boosting'], 'gradient_boosting.pkl')
joblib.dump(trained['Linear Regression'], 'linear_regression.pkl')
print("\n✅ All 3 models saved")

In [ ]:
# STEP 7: COMPARE MODELS — Bar Chart
names  = list(results.keys())
r2s    = [results[n]['r2_test']  for n in names]
maes   = [results[n]['mae']      for n in names]
accs   = [results[n]['accuracy'] for n in names]
colors = ['#3b82f6','#f59e0b','#22c55e']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, vals, title, ylabel in zip(
    axes,
    [r2s, maes, accs],
    ['R² Score (Test)', 'MAE kg/are (Test)', 'Accuracy % (Test)'],
    ['R²','MAE (kg/are)','Accuracy (%)']
):
    bars = ax.bar(names, vals, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(vals)*1.2)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.02,
                f'{val:.3f}', ha='center', fontweight='bold', fontsize=11)
    ax.set_xticklabels(names, rotation=15, ha='right')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# Select best
best_name = max(results, key=lambda k: results[k]['r2_test'])
print(f"\n{'='*50}")
print(f"  BEST MODEL : {best_name}")
print(f"  R² Test    : {results[best_name]['r2_test']:.4f}")
print(f"  MAE        : {results[best_name]['mae']:.4f} kg/are")
print(f"  Accuracy   : {results[best_name]['accuracy']:.1f}%")
print(f"{'='*50}")

In [ ]:
# STEP 7: SELECT AND SAVE BEST MODEL
best_mdl = trained[best_name]
joblib.dump(best_mdl, 'best_model.pkl')
print(f"✅ Saved best_model.pkl  →  {best_name}")

# Feature importance plot
if hasattr(best_mdl, 'feature_importances_'):
    imp_df = pd.DataFrame({'feature': FEATURES, 'importance': best_mdl.feature_importances_})
    imp_df = imp_df.sort_values('importance', ascending=True).tail(15)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors_imp = ['#22c55e' if v > 0.05 else '#94a3b8' for v in imp_df['importance']]
    ax.barh(imp_df['feature'], imp_df['importance']*100, color=colors_imp)
    ax.set_xlabel('Importance (%)', fontsize=12)
    ax.set_title(f'{best_name} — Feature Importance (Top 15)', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("\nTop 10 Features:")
    for _, row in imp_df.sort_values('importance',ascending=False).head(10).iterrows():
        print(f"  {row['feature']:40s}: {row['importance']*100:.1f}%")

In [ ]:
# STEP 8: SAVE METADATA
crop_benchmarks = {
    c: round(float(harvest[harvest.Crop_Type==c]['Yield_Kg_per_Are'].mean()), 4)
    for c in ['Maize','Beans','Rice']
}

metadata = {
    'best_model'  : best_name,
    'features'    : FEATURES,
    'target'      : TARGET,
    'units'       : {'farm_size':'are','area_planted':'are','yield':'kg/are',
                     'note':'1 ha = 100 are — model trained on are units'},
    'crops'       : sorted(harvest.Crop_Type.unique().tolist()),
    'seasons'     : sorted(harvest.Season.unique().tolist()),
    'sectors'     : sorted(harvest.Sector.unique().tolist()),
    'yield_stats' : {
        'mean': round(float(y.mean()),4), 'std': round(float(y.std()),4),
        'min' : round(float(y.min()),4),  'max': round(float(y.max()),4),
        'unit': 'kg/are'
    },
    'crop_benchmarks_kg_are': crop_benchmarks,
    'model_comparison': results,
    '_perf': {k: {'r2': v['r2_test']} for k,v in results.items()},
    'label_encoder_classes': {k: v.classes_.tolist() for k,v in le_dict.items()},
    'r2_score': results[best_name]['r2_test'],
}

with open('model_metadata.json','w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Saved model_metadata.json")
print(f"\nCrop benchmarks (kg/are):")
for k,v in crop_benchmarks.items():
    print(f"  {k}: {v:.2f} kg/are = {v*100:.0f} kg/ha")

print(f"\nFiles saved:")
files = ['best_model.pkl','random_forest.pkl','gradient_boosting.pkl',
         'linear_regression.pkl','label_encoders.pkl','scaler.pkl','model_metadata.json']
for f in files:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f"  {f:35s} {size:,} bytes")
    
print("\n✅ ALL DONE — Ready for deployment!")